In [1]:
import json
import numpy as np
import pandas as pd

#### Single file

In [2]:
# # results_file = "../leaderboard"
# # results_file = "../leaderboard-bu-llama-phi-falcon"
# # results_file = "../leaderboard_t4_all_bench_2"
# # results_file = "../leaderboard_v100_all_bench_2"
# # results_file = "../leaderboard_t4_final_run_2_falcon_old"
# # results_file = "../leaderboard_t4_final_run_50"
# # results_file = "../results/leaderboard_results/leaderboard_gossa_final_run_50"
# # results_file = "../leaderboard"
# results_file = "leaderboard_final"
# # results_file = "../leaderboard_gemma_large"

In [3]:
# results = json.loads(open(results_file, "rb").read())
# # results       

#### Combine many

In [4]:
shared_folder = "/home/azureuser/cloudfiles/code/shared_data/results"

In [5]:
os.listdir("..")
results_files = [
    'leaderboard_eurollm_large', 'leaderboard_eurollm_small',
     'leaderboard_falcon_small',
     # 'leaderboard_gemma_large',
     'leaderboard_gemma_large_gossa', 'leaderboard_gemma_small', 'leaderboard_gemma_small_summary', 'leaderboard_gemma_small_tiny',
     'leaderboard_gpt4o', 'leaderboard_gpt4o_mini',
     'leaderboard_llama_small',
     'leaderboard_mistral_tiny', 'leaderboard_mistral_small', 'leaderboard_mistral_small_summary',
     'leaderboard_olmo_large', 'leaderboard_olmo_small', 'leaderboard_olmo_large_summary',
     'leaderboard_phi_mini',
     'leaderboard_qwen_large', 'leaderboard_qwen_small', 'leaderboard_qwen_small_nothinking', 'leaderboard_qwen_large_nothinking',
     'leaderboard_tinyllama',
]

In [6]:
results = sum([json.loads(open(f"{shared_folder}/{file}", "rb").read()) for file in results_files], [])

# Add costs

In [31]:
from collections import defaultdict
import tiktoken
from datetime import datetime
import sys

# Included benchmarks
costs_included_benchmarks = [
    "AmsterdamSimplification-detailed",
    "INT_Duidelijke_Taal-detailed",
    "CNNDailyMail",
    "XSum",
]

# API pricing table ($ per 1k tokens)
api_model_pricing = {
    "gpt-4o": {"input": 0.005, "output": 0.02},
    "gpt-4o-mini": {"input": 0.0006, "output": 0.0024},
}

# Current hourly GPU rates on Azure
gpu_hourly_rates = {
    "Tesla T4": 0.66,
    "NVIDIA H100 NVL": 9.08,
    "Tesla V100-PCIE-16GB": 3.82,
}

def parse_duration(start, end):
    """Parse duration in seconds per benchmark run."""
    fmt = "%Y-%m-%dT%H:%M:%SZ"
    start_time = datetime.strptime(start, fmt)
    end_time = datetime.strptime(end, fmt)
    return (end_time - start_time).total_seconds()

def count_tokens(model_name, text):
    """Count tokens using tiktoken for a given model and text."""
    try:
        enc = tiktoken.encoding_for_model(model_name)
        return len(enc.encode(text))
    except Exception:
        return 0

results_by_model = defaultdict(lambda: {"total_cost": 0.0, "num_entries": 0})

def get_costs(entry):
    # Process entries in leaderboard dataframe
    try:
        metadata = entry["metadata"]
        model = metadata["llm"]["model_name"]
        benchmark = metadata["benchmark"]["name"]
        if benchmark not in costs_included_benchmarks: return -1

        n_tokens = metadata.get("n_tokens")
        run_output = entry.get("benchmark_results", {}).get("run_output", [])
        n_samples = metadata.get("n_samples", 1)
        run = metadata.get("run")
        device = run.get("system", {}).get("device_info", {}).get("gpu", {}).get("device_name")
        start_time, end_time = run.get("timestamp_bench_start"), run.get("timestamp_bench_end")

        if model in api_model_pricing:
            pricing = api_model_pricing[model]
            n_input = n_tokens.get("n_input_tokens") if isinstance(n_tokens, dict) else None
            n_output = n_tokens.get("n_output_tokens") if isinstance(n_tokens, dict) else None

            if n_input is None or n_output is None:
                inputs = [metadata["benchmark"]["prompt_template"] + (r.get("prompt") or r.get("source") or "") for r in run_output]
                outputs = [r.get("response") for r in run_output]
                n_input = sum(count_tokens(model, p) for p in inputs if isinstance(p, str))
                n_output = sum(count_tokens(model, o) for o in outputs if isinstance(o, str))
                print(benchmark, sum([len(input.split(" ")) for input in inputs]), len(outputs))
            cost = n_input * pricing["input"] / 1000 + n_output * pricing["output"] / 1000

        else:
            duration = parse_duration(start_time, end_time)
            gpu_rate = gpu_hourly_rates.get(device)
            if duration and gpu_rate:
                cost = duration * gpu_rate / 3600
            else:
                return -1
        # results_by_model[model]["total_cost"] += cost
        # results_by_model[model]["num_entries"] += 1
        return cost / n_samples
    except Exception as e:
        print(f"Skipping due to error: {e}")
        return -1

# avg_costs = {}
# for model, data in results_by_model.items():
#     if data["num_entries"] > 0:
#         avg_costs[model] = round(data["total_cost"] / data["num_entries"], 6)

# cost_series = pd.Series(avg_costs, name=("cost", "avg_cost_per_prompt"))
# cost_series

In [32]:
def get_score(entry):
    if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]:
        return entry["benchmark_results"]["score"]["acc"]
    elif entry["metadata"]["benchmark"]["name"] in ["TinyMMLU", "TinyARC", "TinyTruthfulQA"]:
        # return entry["benchmark_results"]["score"]["acc"]
        # return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["irt"],
        # return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["pirt"],
        return entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["gpirt"]
        # return gpirt if not np.isnan(gpirt) else -1
        # return gpirt
    elif entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]:
        return entry["benchmark_results"]["score"]["sari"]["sari"]
    elif entry["metadata"]["benchmark"]["name"] in ["CNNDailyMail", "XSum"]:
        # bert_score = entry["benchmark_results"]["score"]["bert_score"]
        # return np.mean(bert_score["f1"]) if bert_score else 0
        bert_score = entry["benchmark_results"]["score"]["bert_score"]
        return np.mean(bert_score["f1"]) if bert_score else 0
    else:
        return -1

        #     entry["benchmark_results"]["score"]["acc"] if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]
        #     else entry["benchmark_results"]["score"]["sari"]["sari"] if entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]
        #     else np.mean(entry["benchmark_results"]["score"]["bert_score"]["f1"])


filtered_data = [
    {
        "runtime": entry["metadata"]["run"]["time_bench_total"],
        "llm_name": entry["metadata"]["llm"]["model_name"],
        "bench_name": entry["metadata"]["benchmark"]["name"],
        "environment_info_co2": entry["metadata"]["code_carbon"]["emissions"] if entry["metadata"]["code_carbon"] else -1,
        "duration": entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
        "environment_info_energy": entry["metadata"]["code_carbon"]["energy_consumed"] if entry["metadata"]["code_carbon"] else -1,
        # "score":
        #     entry["benchmark_results"]["score"]["acc"] if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]
        #     else entry["benchmark_results"]["score"]["sari"]["sari"] if entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]
        #     else np.mean(entry["benchmark_results"]["score"]["bert_score"]["f1"])
        "score": get_score(entry),
        "costs": get_costs(entry),
        "energy_per_time": entry["metadata"]["code_carbon"]["energy_consumed"] / entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
    } 
    # for entry in results
    for entry in results if "ARCHIVE" not in entry["metadata"]["llm"]["model_name"]
]

INT_Duidelijke_Taal-detailed 6170 100
AmsterdamSimplification-detailed 6380 100
CNNDailyMail 59044 100
XSum 43391 100
INT_Duidelijke_Taal-detailed 6170 100
AmsterdamSimplification-detailed 6380 100
CNNDailyMail 59044 100
XSum 43391 100


In [9]:
# filtered_data

In [10]:
df = pd.DataFrame(filtered_data)

# Pivot the DataFrame to create a multi-level column index
pivot_df = df.pivot_table(
    index='llm_name',
    columns='bench_name',
    values=['score', 'environment_info_co2', 'runtime', 'duration', 'environment_info_energy', 'costs', 'energy_per_time'],
    aggfunc='first'
)

# Reorder the columns to have a multi-level index
pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)

In [11]:
# pivot_df = pivot_df.join(cost_series, how="left")

In [12]:
pivot_df[['costs', 'energy_per_time', 'environment_info_energy', 'duration']]

costs               \
                         AmsterdamSimplification-detailed CNNDailyMail   
llm_name                                                                 
eurollm-22b-instruct                             0.002320     0.011804   
eurollm-9b-instruct                              0.001488     0.008046   
falcon3-7b-instruct                              0.001690     0.005347   
gemma-12b-instruct                               0.007441     0.014805   
gemma-27b-instruct                               0.032587     0.033697   
gpt-4o                                           0.001100     0.006886   
gpt-4o-mini                                      0.000133     0.000815   
llama-3.1-8b-instruct                            0.001841     0.006709   
mistral-7b-instruct-v0.3                         0.002674     0.007869   
mistral-small-instruct                           0.001942     0.008424   
olmo-32b-instruct                                0.004263     0.021086   
olmo-7b-instruct                                 0.002699     0.009963   
phi-4-mini-instruct                              0.008777     0.007617   
qwen-32b                                         0.003657     0.016092   
qwen-32b-THINKING                                0.021237     0.021540   
qwen-8b                                          0.002018     0.008828   
qwen-8b-THINKING                                 0.012258     0.012081   
tiny-llama                                       0.003506     0.006079   

                                                                        \
                         INT_Duidelijke_Taal-detailed TinyARC TinyMMLU   
llm_name                                                                 
eurollm-22b-instruct                         0.002371    -1.0     -1.0   
eurollm-9b-instruct                          0.001917    -1.0     -1.0   
falcon3-7b-instruct                          0.001816    -1.0     -1.0   
gemma-12b-instruct                           0.010114    -1.0     -1.0   
gemma-27b-instruct                           0.033470    -1.0     -1.0   
gpt-4o                                       0.001063    -1.0     -1.0   
gpt-4o-mini                                  0.000130    -1.0     -1.0   
llama-3.1-8b-instruct                        0.001917    -1.0     -1.0   
mistral-7b-instruct-v0.3                     0.002976    -1.0     -1.0   
mistral-small-instruct                       0.002119    -1.0     -1.0   
olmo-32b-instruct                            0.003809    -1.0     -1.0   
olmo-7b-instruct                             0.002119    -1.0     -1.0   
phi-4-mini-instruct                          0.009257    -1.0     -1.0   
qwen-32b                                     0.003556    -1.0     -1.0   
qwen-32b-THINKING                            0.021767    -1.0     -1.0   
qwen-8b                                      0.002220    -1.0     -1.0   
qwen-8b-THINKING                             0.012687    -1.0     -1.0   
tiny-llama                                   0.003077    -1.0     -1.0   

                                                   \
                         TinyTruthfulQA      XSum   
llm_name                                            
eurollm-22b-instruct               -1.0  0.007743   
eurollm-9b-instruct                -1.0  0.004288   
falcon3-7b-instruct                -1.0  0.004565   
gemma-12b-instruct                 -1.0  0.008424   
gemma-27b-instruct                 -1.0  0.033520   
gpt-4o                             -1.0  0.004512   
gpt-4o-mini                        -1.0  0.000572   
llama-3.1-8b-instruct              -1.0  0.004994   
mistral-7b-instruct-v0.3           -1.0  0.007869   
mistral-small-instruct             -1.0  0.004641   
olmo-32b-instruct                  -1.0  0.018135   
olmo-7b-instruct                   -1.0  0.009938   
phi-4-mini-instruct                -1.0  0.007113   
qwen-32b                           -1.0  0.009761   
qwen-32b-THINKING                  -1.0  

In [13]:
# order = ["ARC-NL", "MMLU-NL", "INT_Duidelijke_Taal-detailed", "INT_Duidelijke_Taal-simple", "AmsterdamSimplification-detailed", "AmsterdamSimplification-simple", "CNNDailyMail", "XSum"]
# order = ["MMLU-NL", "ARC-NL", "TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]
order = ["TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]

In [14]:
pivot_df["score"].reindex(columns=order)

,TinyMMLU,TinyARC,TinyTruthfulQA,INT_Duidelijke_Taal-detailed,AmsterdamSimplification-detailed,CNNDailyMail,XSum
llm_name,,,,,,,
eurollm-22b-instruct,0.431427,0.577841,0.508384,38.827863,38.552826,0.647698,0.649587
eurollm-9b-instruct,0.392877,0.584741,0.385861,38.877734,40.720558,0.649766,0.657533
falcon3-7b-instruct,0.412126,0.522075,0.392499,38.987092,39.245750,0.646484,0.641361
gemma-12b-instruct,0.535311,0.688468,0.655981,40.311373,40.782701,0.647721,0.654475
gemma-27b-instruct,0.596027,0.826429,0.631800,40.134874,40.943857,0.647577,0.652869
gpt-4o,0.678687,0.797592,0.725110,39.393222,41.470832,0.646980,0.658448
gpt-4o-mini,0.571936,0.785351,0.612299,38.213730,39.059379,0.625860,0.653440
llama-3.1-8b-instruct,0.476507,0.627029,0.515184,38.718881,39.812963,0.653324,0.659105
mistral-7b-instruct-v0.3,0.449968,0.541196,0.527956,38.851601,41.377323,0.652280,0.636364


# Environmental Impact

In [15]:
pivot_df["environment_info_co2"].reindex(columns=order) * 1000

,TinyMMLU,TinyARC,TinyTruthfulQA,INT_Duidelijke_Taal-detailed,AmsterdamSimplification-detailed,CNNDailyMail,XSum
llm_name,,,,,,,
eurollm-22b-instruct,0.267107,0.109398,0.044709,0.469623,0.537863,2.961309,1.916650
eurollm-9b-instruct,0.568748,0.634980,0.188354,0.272469,0.269864,1.562183,0.827236
falcon3-7b-instruct,0.029235,0.025592,0.026468,0.271353,0.317039,1.097196,0.936111
gemma-12b-instruct,2.373630,1.909131,1.908448,2.104179,1.821158,2.797520,2.015438
gemma-27b-instruct,6.955676,6.982680,6.994607,6.762640,6.708709,7.076838,7.049215
gpt-4o,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000
gpt-4o-mini,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000
llama-3.1-8b-instruct,0.204682,0.193132,0.052069,0.276612,0.343638,1.368424,1.010086
mistral-7b-instruct-v0.3,0.319684,0.156818,0.230292,0.478566,0.488118,1.561274,1.539931


In [16]:
pivot_df["environment_info_energy"].reindex(columns=order) * 1000

,TinyMMLU,TinyARC,TinyTruthfulQA,INT_Duidelijke_Taal-detailed,AmsterdamSimplification-detailed,CNNDailyMail,XSum
llm_name,,,,,,,
eurollm-22b-instruct,6.563621,2.688230,1.098633,11.540055,13.216936,72.768365,47.097917
eurollm-9b-instruct,13.975872,15.603385,4.628441,6.695397,6.631371,38.387596,20.327715
falcon3-7b-instruct,0.718402,0.628881,0.650397,6.667958,7.790604,26.961437,23.003096
gemma-12b-instruct,58.327322,46.913158,46.896383,51.706071,44.751384,68.743588,49.525450
gemma-27b-instruct,170.922116,171.585698,171.878774,166.178648,164.853402,173.899446,173.220655
gpt-4o,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000
gpt-4o-mini,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000,-1000.000000
llama-3.1-8b-instruct,5.029647,4.745853,1.279486,6.797198,8.444225,33.626338,24.820897
mistral-7b-instruct-v0.3,7.855617,3.853498,5.658986,11.759823,11.994546,38.365256,37.840781


In [17]:
pivot_df["duration"].reindex(columns=order)

,TinyMMLU,TinyARC,TinyTruthfulQA,INT_Duidelijke_Taal-detailed,AmsterdamSimplification-detailed,CNNDailyMail,XSum
llm_name,,,,,,,
eurollm-22b-instruct,43.577404,17.829193,7.063549,80.398677,91.199457,465.965655,305.097304
eurollm-9b-instruct,120.788942,135.707105,40.316737,59.674088,58.606270,318.261198,169.461139
falcon3-7b-instruct,5.696377,5.142262,5.294878,56.551664,65.167223,209.787261,179.484457
gemma-12b-instruct,461.858834,316.106011,316.048931,383.246143,292.832797,571.800650,332.493515
gemma-27b-instruct,1317.755013,1317.054351,1321.322716,1313.260219,1290.388445,1334.048220,1327.263937
gpt-4o,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
gpt-4o-mini,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
llama-3.1-8b-instruct,41.202962,50.000228,10.749548,58.224740,71.521716,264.440645,195.852013
mistral-7b-instruct-v0.3,66.888424,33.101944,48.852834,103.612214,104.374307,309.764563,310.072911


In [18]:
pivot_df["score"].reindex(columns=order)

,TinyMMLU,TinyARC,TinyTruthfulQA,INT_Duidelijke_Taal-detailed,AmsterdamSimplification-detailed,CNNDailyMail,XSum
llm_name,,,,,,,
eurollm-22b-instruct,0.431427,0.577841,0.508384,38.827863,38.552826,0.647698,0.649587
eurollm-9b-instruct,0.392877,0.584741,0.385861,38.877734,40.720558,0.649766,0.657533
falcon3-7b-instruct,0.412126,0.522075,0.392499,38.987092,39.245750,0.646484,0.641361
gemma-12b-instruct,0.535311,0.688468,0.655981,40.311373,40.782701,0.647721,0.654475
gemma-27b-instruct,0.596027,0.826429,0.631800,40.134874,40.943857,0.647577,0.652869
gpt-4o,0.678687,0.797592,0.725110,39.393222,41.470832,0.646980,0.658448
gpt-4o-mini,0.571936,0.785351,0.612299,38.213730,39.059379,0.625860,0.653440
llama-3.1-8b-instruct,0.476507,0.627029,0.515184,38.718881,39.812963,0.653324,0.659105
mistral-7b-instruct-v0.3,0.449968,0.541196,0.527956,38.851601,41.377323,0.652280,0.636364


In [19]:
pivot_df["duration"]

,AmsterdamSimplification-detailed,CNNDailyMail,INT_Duidelijke_Taal-detailed,TinyARC,TinyMMLU,TinyTruthfulQA,XSum
llm_name,,,,,,,
eurollm-22b-instruct,91.199457,465.965655,80.398677,17.829193,43.577404,7.063549,305.097304
eurollm-9b-instruct,58.606270,318.261198,59.674088,135.707105,120.788942,40.316737,169.461139
falcon3-7b-instruct,65.167223,209.787261,56.551664,5.142262,5.696377,5.294878,179.484457
gemma-12b-instruct,292.832797,571.800650,383.246143,316.106011,461.858834,316.048931,332.493515
gemma-27b-instruct,1290.388445,1334.048220,1313.260219,1317.054351,1317.755013,1321.322716,1327.263937
gpt-4o,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
gpt-4o-mini,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
llama-3.1-8b-instruct,71.521716,264.440645,58.224740,50.000228,41.202962,10.749548,195.852013
mistral-7b-instruct-v0.3,104.374307,309.764563,103.612214,33.101944,66.888424,48.852834,310.072911


In [20]:
pivot_df["duration"].reindex(columns=order)

,TinyMMLU,TinyARC,TinyTruthfulQA,INT_Duidelijke_Taal-detailed,AmsterdamSimplification-detailed,CNNDailyMail,XSum
llm_name,,,,,,,
eurollm-22b-instruct,43.577404,17.829193,7.063549,80.398677,91.199457,465.965655,305.097304
eurollm-9b-instruct,120.788942,135.707105,40.316737,59.674088,58.606270,318.261198,169.461139
falcon3-7b-instruct,5.696377,5.142262,5.294878,56.551664,65.167223,209.787261,179.484457
gemma-12b-instruct,461.858834,316.106011,316.048931,383.246143,292.832797,571.800650,332.493515
gemma-27b-instruct,1317.755013,1317.054351,1321.322716,1313.260219,1290.388445,1334.048220,1327.263937
gpt-4o,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
gpt-4o-mini,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
llama-3.1-8b-instruct,41.202962,50.000228,10.749548,58.224740,71.521716,264.440645,195.852013
mistral-7b-instruct-v0.3,66.888424,33.101944,48.852834,103.612214,104.374307,309.764563,310.072911


In [21]:
# pivot_df["environment_info_co2"] = pivot_df["environment_info_co2"].reindex(columns=order)
pivot_df["score"] = pivot_df["score"].reindex(columns=order)
pivot_df["duration"] = pivot_df["duration"].reindex(columns=order)

### Map to categories

In [22]:
def get_env_cat_emissions(score):
    if score < 0:
        return score
    if score < 0.0010:
        return 5
    elif score < 0.0013:
        return 4
    elif score < 0.0016:
        return 3
    elif score < 0.0020:
        return 2
    else:
        return 1

pivot_df.loc[:, ("environment_info_co2", "co2_emissions_mean")] = pivot_df["environment_info_co2"].mean(axis=1)
pivot_df.loc[:, ("environment_info_co2", "co2_emissions_category")] = list(map(lambda x: get_env_cat_emissions(x), pivot_df["environment_info_co2"]["co2_emissions_mean"]))

In [23]:
def get_env_cat_energy(score):
    if score < 0:
        return score
    if score < 0.015:
        return 5
    elif score < 0.025:
        return 4
    elif score < 0.05:
        return 3
    elif score < 0.1:
        return 2
    else:
        return 1

pivot_df.loc[:, ("environment_info_energy", "energy_use_mean")] = pivot_df["environment_info_energy"].mean(axis=1)
pivot_df.loc[:, ("environment_info_energy", "energy_use_category")] = list(map(lambda x: int(get_env_cat_energy(x)), pivot_df["environment_info_energy"]["energy_use_mean"]))

In [24]:
min_co2 = pivot_df["environment_info_co2"]["co2_emissions_mean"].where(pivot_df["environment_info_co2"]["co2_emissions_mean"] > 0).min()
max_co2 = pivot_df["environment_info_co2"]["co2_emissions_mean"].where(pivot_df["environment_info_co2"]["co2_emissions_mean"] > 0).max()
# pivot_df["environment_info_co2"]["Average CO2"] > 0
max_co2 - min_co2

0.0065467672249321105

In [25]:
min_co2 = pivot_df["environment_info_energy"]["energy_use_mean"].where(pivot_df["environment_info_energy"]["energy_use_mean"] > 0).min()
max_co2 = pivot_df["environment_info_energy"]["energy_use_mean"].where(pivot_df["environment_info_energy"]["energy_use_mean"] > 0).max()
# pivot_df["environment_info_co2"]["Average CO2"] > 0
max_co2 - min_co2

0.16087399496085786

In [26]:
pivot_df["environment_info_co2"][["co2_emissions_mean", "co2_emissions_category"]].join(
pivot_df["environment_info_energy"][["energy_use_mean", "energy_use_category"]])

,co2_emissions_mean,co2_emissions_category,energy_use_mean,energy_use_category
llm_name,,,,
eurollm-22b-instruct,0.000901,5.0,0.022139,4
eurollm-9b-instruct,0.000618,5.0,0.015179,4
falcon3-7b-instruct,0.000386,5.0,0.009489,5
gemma-12b-instruct,0.002133,1.0,0.052409,2
gemma-27b-instruct,0.006933,1.0,0.170363,1
gpt-4o,-1.000000,-1.0,-1.000000,-1
gpt-4o-mini,-1.000000,-1.0,-1.000000,-1
llama-3.1-8b-instruct,0.000493,5.0,0.012106,5
mistral-7b-instruct-v0.3,0.000682,5.0,0.016761,4


# Costs

In [34]:
def get_env_cat_energy(score):
    if score < 0:
        return score
    if score < 0.015:
        return 5
    elif score < 0.025:
        return 4
    elif score < 0.05:
        return 3
    elif score < 0.1:
        return 2
    else:
        return 1

pivot_df.loc[:, ("costs", "costs_mean")] = pivot_df["costs"][costs_included_benchmarks].mean(axis=1) * 1000

In [35]:
pivot_df["costs"]

,AmsterdamSimplification-detailed,CNNDailyMail,INT_Duidelijke_Taal-detailed,TinyARC,TinyMMLU,TinyTruthfulQA,XSum,costs_mean
llm_name,,,,,,,,
eurollm-22b-instruct,0.002320,0.011804,0.002371,-1.0,-1.0,-1.0,0.007743,6.059639
eurollm-9b-instruct,0.001488,0.008046,0.001917,-1.0,-1.0,-1.0,0.004288,3.934667
falcon3-7b-instruct,0.001690,0.005347,0.001816,-1.0,-1.0,-1.0,0.004565,3.354556
gemma-12b-instruct,0.007441,0.014805,0.010114,-1.0,-1.0,-1.0,0.008424,10.196083
gemma-27b-instruct,0.032587,0.033697,0.033470,-1.0,-1.0,-1.0,0.033520,33.318556
gpt-4o,0.001100,0.006886,0.001063,-1.0,-1.0,-1.0,0.004512,3.390225
gpt-4o-mini,0.000133,0.000815,0.000130,-1.0,-1.0,-1.0,0.000572,0.412377
llama-3.1-8b-instruct,0.001841,0.006709,0.001917,-1.0,-1.0,-1.0,0.004994,3.865306
mistral-7b-instruct-v0.3,0.002674,0.007869,0.002976,-1.0,-1.0,-1.0,0.007869,5.347111


# Factuality

In [36]:
# pivot_df.loc[:, ("score", "Reasoning")] = ((pivot_df["score"]["ARC-NL"] + pivot_df["score"]["MMLU-NL"]) / 2).tolist()
pivot_df.loc[:, ("score", "factuality_mean")] = ((pivot_df["score"]["TinyMMLU"] + pivot_df["score"]["TinyARC"] + pivot_df["score"]["TinyTruthfulQA"]) / 3).tolist()

In [37]:
def get_factuality_cat(score):
    if score > 0.8:
        return 5
    elif score > 0.7:
        return 4
    elif score > 0.6:
        return 3
    elif score > 0.5:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    
pivot_df.loc[:, ("score", "factuality_category")] = list(map(lambda x: get_factuality_cat(x), pivot_df["score"]["factuality_mean"]))

In [38]:
# pivot_df["score"][order]
# pivot_df["score"].reindex(columns=order)
# pivot_df["score"][["ARC-NL", "MMLU-NL", "Reasoning", "ReasoningCategory"]]
pivot_df["score"][["TinyMMLU", "TinyARC", "TinyTruthfulQA", "factuality_mean", "factuality_category"]]

,TinyMMLU,TinyARC,TinyTruthfulQA,factuality_mean,factuality_category
llm_name,,,,,
eurollm-22b-instruct,0.431427,0.577841,0.508384,0.505884,2
eurollm-9b-instruct,0.392877,0.584741,0.385861,0.454493,1
falcon3-7b-instruct,0.412126,0.522075,0.392499,0.442233,1
gemma-12b-instruct,0.535311,0.688468,0.655981,0.626586,3
gemma-27b-instruct,0.596027,0.826429,0.631800,0.684752,3
gpt-4o,0.678687,0.797592,0.725110,0.733796,4
gpt-4o-mini,0.571936,0.785351,0.612299,0.656529,3
llama-3.1-8b-instruct,0.476507,0.627029,0.515184,0.539573,2
mistral-7b-instruct-v0.3,0.449968,0.541196,0.527956,0.506374,2


# Simplification

In [39]:
pivot_df.loc[:, ("score", "simplification_mean")] = ((pivot_df["score"]["AmsterdamSimplification-detailed"] + pivot_df["score"]["INT_Duidelijke_Taal-detailed"]) / 2).tolist()

In [40]:
def get_simplification_cat_old(score):
    if score > 40:
        return 5
    elif score > 30:
        return 4
    elif score > 20:
        return 3
    elif score > 10:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1

def get_simplification_cat(score):
    if score > 44:
        return 5
    elif score > 38:
        return 4
    elif score > 32:
        return 3
    elif score > 26:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1


pivot_df.loc[:, ("score", "simplification_category")] = list(map(lambda x: get_simplification_cat(x), pivot_df["score"]["simplification_mean"]))

In [41]:
pivot_df["score"][["AmsterdamSimplification-detailed", "INT_Duidelijke_Taal-detailed", "simplification_mean", "simplification_category"]]

,AmsterdamSimplification-detailed,INT_Duidelijke_Taal-detailed,simplification_mean,simplification_category
llm_name,,,,
eurollm-22b-instruct,38.552826,38.827863,38.690344,4
eurollm-9b-instruct,40.720558,38.877734,39.799146,4
falcon3-7b-instruct,39.245750,38.987092,39.116421,4
gemma-12b-instruct,40.782701,40.311373,40.547037,4
gemma-27b-instruct,40.943857,40.134874,40.539365,4
gpt-4o,41.470832,39.393222,40.432027,4
gpt-4o-mini,39.059379,38.213730,38.636555,4
llama-3.1-8b-instruct,39.812963,38.718881,39.265922,4
mistral-7b-instruct-v0.3,41.377323,38.851601,40.114462,4


# Summarization

In [42]:
pivot_df.loc[:, ("score", "summarization_mean")] = ((pivot_df["score"]["CNNDailyMail"] + pivot_df["score"]["XSum"]) / 2).tolist()

In [43]:
def get_summarization_cat(score):
    if score > 0.65:
        return 5
    elif score > 0.60:
        return 4
    elif score > 0.55:
        return 3
    elif score > 0.50:
        return 2
    elif not np.isnan(score):
        return 1
    else:
        return -1
    
pivot_df.loc[:, ("score", "summarization_category")] = list(map(lambda x: get_summarization_cat(x), pivot_df["score"]["summarization_mean"]))

In [44]:
pivot_df["score"][["CNNDailyMail", "XSum", "summarization_mean", "summarization_category"]]

,CNNDailyMail,XSum,summarization_mean,summarization_category
llm_name,,,,
eurollm-22b-instruct,0.647698,0.649587,0.648643,4
eurollm-9b-instruct,0.649766,0.657533,0.653649,5
falcon3-7b-instruct,0.646484,0.641361,0.643923,4
gemma-12b-instruct,0.647721,0.654475,0.651098,5
gemma-27b-instruct,0.647577,0.652869,0.650223,5
gpt-4o,0.646980,0.658448,0.652714,5
gpt-4o-mini,0.625860,0.653440,0.639650,4
llama-3.1-8b-instruct,0.653324,0.659105,0.656215,5
mistral-7b-instruct-v0.3,0.652280,0.636364,0.644322,4


# All scores

In [45]:
# pd.to_datetime(pivot_df["runtime"]["ARC-NL"]).map(lambda x: x.second)

In [46]:
final_scores_categories = pivot_df["score"][[
    "factuality_mean", "factuality_category",
    "simplification_mean", "simplification_category",
    "summarization_mean", "summarization_category"]].join(
pivot_df["environment_info_energy"][["energy_use_mean", "energy_use_category"]]).join(
pivot_df["costs"][["costs_mean"]])

In [47]:
final_scores_categories

,factuality_mean,factuality_category,simplification_mean,simplification_category,summarization_mean,summarization_category,energy_use_mean,energy_use_category,costs_mean
llm_name,,,,,,,,,
eurollm-22b-instruct,0.505884,2,38.690344,4,0.648643,4,0.022139,4,6.059639
eurollm-9b-instruct,0.454493,1,39.799146,4,0.653649,5,0.015179,4,3.934667
falcon3-7b-instruct,0.442233,1,39.116421,4,0.643923,4,0.009489,5,3.354556
gemma-12b-instruct,0.626586,3,40.547037,4,0.651098,5,0.052409,2,10.196083
gemma-27b-instruct,0.684752,3,40.539365,4,0.650223,5,0.170363,1,33.318556
gpt-4o,0.733796,4,40.432027,4,0.652714,5,-1.000000,-1,3.390225
gpt-4o-mini,0.656529,3,38.636555,4,0.639650,4,-1.000000,-1,0.412377
llama-3.1-8b-instruct,0.539573,2,39.265922,4,0.656215,5,0.012106,5,3.865306
mistral-7b-instruct-v0.3,0.506374,2,40.114462,4,0.644322,4,0.016761,4,5.347111


In [48]:
from llm_eval.language_models.llms import llm_config

/anaconda/envs/env_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [49]:
name_map = {
    model_config["id"].split("/")[1]: model_name
    for model_name, model_config in llm_config.MODEL_MAPPING.items()
}

name_map.update({
    # "gpt-4o": "GPT-4o",
    # "gpt-4o-mini": "GPT-4o-mini"
    "GPT-4o": "gpt-4o",
    "GPT-4o-mini": "gpt-4o-mini"
})

In [50]:
name_map

{'Falcon3-7B-Instruct': 'falcon3-7b-instruct',
 'Mistral-7B-Instruct-v0.3': 'mistral-7b-instruct-v0.3',
 'Mistral-Small-24B-Instruct-2501': 'mistral-small-instruct',
 'Mistral-Large-Instruct-2411': 'mistral-large-instruct',
 'Mistral-Large-Instruct-2407-GPTQ': 'mistral-large-instruct-quantized',
 'TinyLlama-1.1B-Chat-v1.0': 'tiny-llama',
 'Llama-3.2-3B-Instruct': 'llama-3.2-3b-instruct',
 'Llama-3.1-8B-Instruct': 'llama-3.1-8b-instruct',
 'Llama-3.3-70B-Instruct': 'llama-3.3-70b-instruct',
 'Phi-4-mini-instruct': 'phi-4-mini-instruct',
 'OLMo-2-1124-7B-Instruct': 'olmo-7b-instruct',
 'OLMo-2-0325-32B-Instruct': 'olmo-32b-instruct',
 'EuroLLM-9B-Instruct': 'eurollm-9b-instruct',
 'EuroLLM-22B-Instruct-Preview': 'eurollm-22b-instruct',
 'Qwen3-8B': 'qwen-8b',
 'Qwen3-32B': 'qwen-32b',
 'gemma-3-12b-it': 'gemma-12b-instruct',
 'gemma-3-27b-it': 'gemma-27b-instruct',
 'GPT-4o': 'gpt-4o',
 'GPT-4o-mini': 'gpt-4o-mini'}

In [51]:
import json
existing_data = json.load(open("../llm-eval-website/_data/models.json", "r"))
existing_data[0]

{'model': 'TinyLlama-1.1B-Chat-v1.0',
 'model_url': 'https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'provider': 'StatNLP',
 'provider_url': 'https://www.sutd.edu.sg/',
 'location': 'Singapore',
 'location_link': 'https://www.sutd.edu.sg/about',
 'license': 'open',
 'license_name': 'Apache License 2.0',
 'license_link': 'https://huggingface.co/datasets/choosealicense/licenses/blob/main/markdown/apache-2.0.md',
 'data_openness': 'open',
 'data_link': 'https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0#this-model',
 'data_datasets': 'pretraining: SlimPajama (=50% RedPajama) + StarCoder (86 programming languages); chat: UltraChat (generated by ChatGPT) + UltraFeedback (generated by different models); All links https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'data_dutch': 'mostly English; possibly via Wikipedia',
 'data_privacy': 'dependency: synthetic data using other models',
 'data_cutoff': '2023',
 'costs': '4€/h for a VM',
 'in_short': 'TBA',
 'use_cas

In [55]:
aspects = ["factuality", "simplification", "summarization", "energy_use", "costs"]

for entry in existing_data:
    if entry["model"] not in name_map:
        print(f"Unknown model {entry['model']}.")
        continue
    model_name_simple = name_map[entry["model"]]
    if model_name_simple not in final_scores_categories.index:
        print(f"Missing new {model_name_simple} scores!!! Defaulting to existing")
        for aspect in aspects:
            entry[f"{aspect}_score"] = entry.pop(f"{aspect}_score", -1)
            entry[f"{aspect}"] = entry.pop(aspect, -1)
        continue
    for aspect in aspects:
        entry.pop(aspect, -1)
        if f"{aspect}_mean" in final_scores_categories.columns:
            entry[f"{aspect}_score"] = float(final_scores_categories.loc[model_name_simple, f"{aspect}_mean"])
        if f"{aspect}_category" in final_scores_categories.columns:
            entry[f"{aspect}"] = int(final_scores_categories.loc[model_name_simple, f"{aspect}_category"])

Missing new mistral-large-instruct scores!!! Defaulting to existing
Unknown model TechxGenus/Mistral-Large-Instruct-2407-GPTQ.
Missing new llama-3.2-3b-instruct scores!!! Defaulting to existing
Missing new llama-3.3-70b-instruct scores!!! Defaulting to existing


In [192]:
# existing_data

In [54]:
json.dump(existing_data, open("../llm-eval-website/_data/models.json", "w"), indent=4,  ensure_ascii=False)

### Total runtime

In [ ]:
seconds = sum([
    pd.to_datetime(pivot_df["runtime"][column]).map(lambda x: x.second).sum()
    for column in pivot_df["runtime"].columns
])

In [ ]:
print(f"{seconds} seconds // {seconds // 60} minutes")

624 seconds // 10 minutes


In [ ]:
import pandas as pd

# Assuming pivot_df is your DataFrame and it has a 'runtime' column which is a DataFrame itself
for column in pivot_df["runtime"].columns:
    # # Convert the column to datetime
    datetime_series = pd.to_datetime(pivot_df["runtime"][column])
    seconds_series = datetime_series.map(lambda x: x.second)
    # # Create a new column in pivot_df using .loc
    pivot_df.loc[:, f"runtime-{column}-seconds"] = seconds_series

In [ ]:
pivot_df["score"]


,ARC-NL,AmsterdamSimplification-detailed,INT_Duidelijke_Taal-detailed,MMLU-NL
llm_name,,,,
gpt-4o,0.95,NaN,0.245833,0.67
gpt-4o-mini,0.90,NaN,0.271299,NaN


### Color

In [23]:
pivot_df.columns = pd.MultiIndex.from_tuples(pivot_df.columns)

# Define a function to apply conditional formatting
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: lightgreen' if v else '' for v in is_max]
    # return ['background-color: lightcoral' if v else '' for v in is_max]

def highlight_min(s):
    is_min = s == s.min()
    return ['background-color: lightgreen' if v else '' for v in is_min]

# Apply the conditional formatting
# styled_df = pivot_df.style.apply(highlight_max, subset=['runtime'], axis=0)
# styled_df = styled_df.apply(highlight_min, subset=['environment_info_co2'], axis=0)
styled_df = pivot_df.style.apply(highlight_min, subset=["runtime"])
styled_df = styled_df.apply(highlight_max, subset=["score"])

# Display the styled DataFrame
styled_df

In [24]:
order = ["ARC-NL", "MMLU-NL", "INT_Duidelijke_Taal-detailed", "INT_Duidelijke_Taal-simple"]
# order = ["ARC-NL", "MMLU-NL", "INT_Duidelijke_Taal-simple"]

In [25]:
# pivot_df[["score"]].style.background_gradient(axis=None, cmap='YlGn')
pivot_df["score"].reindex(columns=order).style.background_gradient(axis=None, cmap='YlGn')

,ARC-NL,MMLU-NL,INT_Duidelijke_Taal-detailed,INT_Duidelijke_Taal-simple
llm_name,,,,
falcon-7b-instruct,0.220000,0.240000,0.000000,0.000000
gpt-4o,1.000000,0.660000,0.251022,0.238183
mistral-7b-instruct-v0.3,0.740000,0.460000,0.031726,0.044986
phi-4-mini-instruct,0.680000,0.520000,0.012791,0.018556
tiny-llama,0.160000,0.200000,0.006404,0.000000


In [26]:
max_impact = pivot_df["environment_info_co2"].max().max()

In [183]:
max_impact

0.0011103165668545619

In [29]:

for bench in pivot_df["environment_info_co2"].columns:
    # pivot_df.loc["gpt-4o", ("environment_info_co2", bench)] = 10*max_impact
    pivot_df.loc["gpt-4o", ("environment_info_co2", bench)] = np.NaN

In [30]:
# pivot_df[["environment_info_co2"]].style.background_gradient(axis=None, cmap='YlOrRd', vmin=0)
pivot_df["environment_info_co2"].reindex(columns=order).style.background_gradient(axis=None, cmap='YlOrRd', vmin=0)

,ARC-NL,MMLU-NL,INT_Duidelijke_Taal-detailed,INT_Duidelijke_Taal-simple
llm_name,,,,
falcon-7b-instruct,0.000175,0.000604,0.000999,0.001102
gpt-4o,nan,nan,nan,nan
mistral-7b-instruct-v0.3,0.000497,0.001033,0.000688,0.000664
phi-4-mini-instruct,0.001095,0.001110,0.000665,0.000717
tiny-llama,0.000373,0.000420,0.000379,0.000461


In [126]:
pivot_df

environment_info_co2                               \
                                       ARC-NL INT_Duidelijke_Taal-detailed   
llm_name                                                                     
falcon-7b-instruct                   0.000003                     0.000015   
gpt-4o                               0.005005                     0.005005   
mistral-7b-instruct-v0.3             0.000024                     0.000025   
phi-4-mini-instruct                  0.000049                     0.000030   
tiny-llama                           0.000009                     0.000019   

                                                                      runtime  \
                         INT_Duidelijke_Taal-simple   MMLU-NL          ARC-NL   
llm_name                                                                        
falcon-7b-instruct                         0.000040  0.000500  0:00:00.881131   
gpt-4o                                     0.005005  0.005005  0:00:00.774783   
mistral-7b-instruct-v0.3                   0.000022  0.000182  0:00:09.955108   
phi-4-mini-instruct                        0.000032  0.000132  0:00:23.564705   
tiny-llama                                 0.000019  0.000058  0:00:04.983807   

                                                       \
                         INT_Duidelijke_Taal-detailed   
llm_name                                                
falcon-7b-instruct                     0:00:06.524728   
gpt-4o                                 0:00:17.958433   
mistral-7b-instruct-v0.3               0:00:10.796004   
phi-4-mini-instruct                    0:00:14.604679   
tiny-llama                             0:00:10.519186   

                                                                     score  \
                         INT_Duidelijke_Taal-simple         MMLU-NL ARC-NL   
llm_name                                                                     
falcon-7b-instruct                   0:00:16.812516  0:05:08.864577    0.0   
gpt-4o                               0:00:01.727736  0:00:00.598899    1.0   
mistral-7b-instruct-v0.3             0:00:09.839951  0:01:49.177146    0.5   
phi-4-mini-instruct                  0:00:16.006698  0:01:16.785089    1.0   
tiny-llama                           0:00:10.763650  0:00:34.754664    0.0   

                                                       \
                         INT_Duidelijke_Taal-detailed   
llm_name                                                
falcon-7b-instruct                           0.000000   
gpt-4o                                       0.138811   
mistral-7b-instruct-v0.3                     0.000000   
phi-4-mini-instruct                          0.000000   
tiny-llama                                   0.000000   

                                                             
                         INT_Duidelijke_Taal-simple MMLU-NL  
llm_name                                                     
falcon-7b-instruct                              0.0     0.0  
gpt-4o                                          0.0     1.0  
mistral-7b-instruct-v0.3                        0.0     1.0  
phi-4-mini-instruct                             0.0     0.5  
tiny-llama                                      0.0     0.0

### Inspect results

In [3]:
# [entry["benchmark_results"] for entry in results if entry["metadata"]["benchmark"]["name"] == "ARC-NL"]

### Check TinyBenchmarks

In [1]:
import json
import numpy as np
import pandas as pd

In [2]:
# results_file = "../leaderboard"
results_file_nl = "../results/results_tinybenchmarks/leaderboard_tinybenches_NL_run2"
results_file_en = "../results/results_tinybenchmarks/leaderboard_tinybenches_EN"

In [3]:
results_nl = json.loads(open(results_file_nl, "rb").read())
results_en = json.loads(open(results_file_en, "rb").read())
# results

In [4]:
filtered_data_nl = [
    {
        "runtime": entry["metadata"]["run"]["time_bench_total"],
        "llm_name": entry["metadata"]["llm"]["model_name"],
        "bench_name": entry["metadata"]["benchmark"]["name"],
        "language": entry["metadata"]["benchmark"]["language"],
        "environment_info_co2": entry["metadata"]["code_carbon"]["emissions"] if entry["metadata"]["code_carbon"] else -1,
        "duration": entry["metadata"]["code_carbon"]["duration"] if entry["metadata"]["code_carbon"] else -1,
        # "score":
        #     entry["benchmark_results"]["score"]["acc"] if entry["metadata"]["benchmark"]["name"] in ["MMLU-NL", "ARC-NL"]
        #     else entry["benchmark_results"]["score"]["sari"]["sari"] if entry["metadata"]["benchmark"]["name"] in ["INT_Duidelijke_Taal-detailed", "AmsterdamSimplification-detailed"]
        #     else np.mean(entry["benchmark_results"]["score"]["bert_score"]["f1"])
        # "score": get_score(entry),
        "acc": entry["benchmark_results"]["score"]["acc"],
        "irt": entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["irt"],
        "pirt": entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["pirt"],
        "gpirt": entry["benchmark_results"]["score"]["tiny_scores"][entry["metadata"]["benchmark"]["name"].removeprefix("Tiny").lower()]["gpirt"],
    } 
    for entry in results_nl + results_en
]

In [5]:
df = pd.DataFrame(filtered_data_nl).sort_values(["bench_name", "llm_name"])

In [6]:
def get_reasoning_cat(score):
    if score > 0.8:
        return 5
    elif score > 0.7:
        return 4
    elif score > 0.6:
        return 3
    elif score > 0.5:
        return 2
    else:
        return 1

for score in ["acc", "irt", "pirt", "gpirt"]:
    df[f"{score}-cat"] = list(map(lambda x: get_reasoning_cat(x), df[score]))

In [7]:
# df

In [24]:
scores_per_model = df[["language", "llm_name", "acc", "irt", "pirt", "gpirt"]].groupby(["language", "llm_name"]).agg("mean")

In [25]:
for score in ["acc", "irt", "pirt", "gpirt"]:
    scores_per_model[f"{score}-cat"] = list(map(lambda x: get_reasoning_cat(x), scores_per_model[score]))

In [26]:
scores_per_model

acc       irt      pirt     gpirt  \
language llm_name                                                           
EN       falcon3-7b-instruct       0.660000  0.645539  0.626685  0.622339   
         gpt-4o                    0.886667  0.872474  0.781064  0.804992   
         gpt-4o-mini               0.770000  0.748699  0.726476  0.734314   
         llama-3.1-8b-instruct     0.650000  0.646305  0.609371  0.617748   
         mistral-7b-instruct-v0.3  0.640000  0.643128  0.609786  0.620171   
         phi-4-mini-instruct       0.650000  0.626894  0.596922  0.606930   
         tiny-llama                0.200000  0.205237  0.271875  0.245840   
NL       falcon3-7b-instruct       0.456667  0.445069  0.476974  0.465071   
         gpt-4o                    0.843333  0.818813  0.764463  0.779532   
         gpt-4o-mini               0.750000  0.733966  0.710063  0.716339   
         llama-3.1-8b-instruct     0.580000  0.601474  0.551163  0.566957   
         mistral-7b-instruct-v0.3  0.573333  0.588736  0.565828  0.571738   
         phi-4-mini-instruct       0.573333  0.589339  0.552433  0.561222   
         tiny-llama                0.203333  0.209361  0.275678  0.266239   

                                   acc-cat  irt-cat  pirt-cat  gpirt-cat  
language llm_name                                                         
EN       falcon3-7b-instruct             3        3         3          3  
         gpt-4o                          5        5         4          5  
         gpt-4o-mini                     4        4         4          4  
         llama-3.1-8b-instruct           3        3         3          3  
         mistral-7b-instruct-v0.3        3        3         3          3  
         phi-4-mini-instruct             3        3         2          3  
         tiny-llama                      1        1         1          1  
NL       falcon3-7b-instruct             1        1         1          1  
         gpt-4o                          5        5         4          4  
         gpt-4o-mini                     4        4         4          4  
         llama-3.1-8b-instruct           2        3         2          2  
         mistral-7b-instruct-v0.3        2        2         2          2  
         phi-4-mini-instruct             2        2         2          2  
         tiny-llama                      1        1         1          1

In [28]:
df.sort_values(["llm_name", "language"])

,runtime,llm_name,bench_name,language,environment_info_co2,duration,acc,irt,pirt,gpirt,acc-cat,irt-cat,pirt-cat,gpirt-cat
31,0:01:53.020717,falcon3-7b-instruct,TinyARC,EN,0.000184,112.472514,0.81,0.781570,0.673968,0.699768,5,4,3,3
30,0:02:37.077843,falcon3-7b-instruct,TinyMMLU,EN,0.000256,156.399610,0.63,0.650763,0.642587,0.643494,3,3,3,3
32,0:02:23.135655,falcon3-7b-instruct,TinyTruthfulQA,EN,0.000234,142.714425,0.54,0.504284,0.563499,0.523756,2,2,2,2
10,0:01:45.969907,falcon3-7b-instruct,TinyARC,NL,0.000172,105.447404,0.51,0.473549,0.491253,0.487008,2,1,1,1
9,0:02:21.905029,falcon3-7b-instruct,TinyMMLU,NL,0.000231,141.208344,0.45,0.472429,0.509726,0.505588,1,1,2,2
11,0:01:46.458370,falcon3-7b-instruct,TinyTruthfulQA,NL,0.000173,105.989146,0.41,0.389229,0.429942,0.402617,1,1,1,1
25,0:00:46.628860,gpt-4o,TinyARC,EN,-1.000000,-1.000000,0.95,0.946246,0.793201,0.829897,5,5,4,5
24,0:00:30.197598,gpt-4o,TinyMMLU,EN,-1.000000,-1.000000,0.84,0.847431,0.764877,0.774036,5,5,4,4
26,0:00:30.176743,gpt-4o,TinyTruthfulQA,EN,-1.000000,-1.000000,0.87,0.823745,0.785114,0.811042,5,5,4,5
4,0:00:26.195727,gpt-4o,TinyARC,NL,-1.000000,-1.000000,0.93,0.910410,0.770173,0.803798,5,5,4,5


In [10]:
from collections import defaultdict

results = defaultdict(dict)

for bench_en, bench_nl in zip(results_en, results_nl):
    model = bench_en["metadata"]["llm"]["model_name"]
    bench = bench_en["metadata"]["benchmark"]["name"]
    for ind, (result_en, result_nl) in enumerate(zip(bench_en["benchmark_results"]["run_output"], bench_nl["benchmark_results"]["run_output"])):
        # print(result_en["input"])
        # print(result_nl["input"])
        results[f"{bench}-{ind}"]["index"] = ind
        results[f"{bench}-{ind}"]["bench"] = bench
        results[f"{bench}-{ind}"]["EN"] = result_en["input"]
        results[f"{bench}-{ind}"]["NL"] = result_nl["input"]
        results[f"{bench}-{ind}"]["target"] = result_en["target"]
        results[f"{bench}-{ind}"][f"{model}-en"] = result_en["response"]
        results[f"{bench}-{ind}"][f"{model}-nl"] = result_nl["response"]
    #     break
    # break

In [11]:
pd.DataFrame(results.values()).to_csv("tinyBenchmarkResults.csv")

In [392]:
qwen = json.load(open("../leaderboard_qwen_small", "r"))

In [393]:
[results for results in qwen if results["metadata"]["benchmark"]["name"] == "CNNDailyMail"][0].keys()

In [394]:
# from pprint import pprint
# pprint([entry["response"]
for entry in [results for results in qwen if results["metadata"]["benchmark"]["name"] == "TinyARC"][0]["benchmark_results"]["run_output"]:
    print(entry["response"])

A
A
A
A
A
A
Okay, let's see. The question is about hardness tests between different substances. The students are testing how hard each substance is by seeing if they can scratch each other. The information given is that X scratches Y, Y scratches Z, and Z scratches W. The question is asking which statement best describes the hardness of substance W.First, I need to recall the concept of hardness in materials. If a material can scratch another, it's harder. So, the order of hardness would be based on the ability to scratch others. So, if X can scratch Y, that means X is harder than Y. Similarly, Y can scratch Z, so Y is harder than Z. Then Z can scratch W, so Z is harder than W. So putting this all together, the order from hardest to softest would be X > Y > Z > W. Wait, but let me make sure. If X scratches Y, then X is harder than Y. Y scratches Z, so
A
A
B
A
A
Okay, let's see. The question is asking which energy source Roy should identify that isn't easily replaceable in nature. The o

In [288]:
# from pprint import pprint
# pprint([entry["response"]
for entry in [results for results in qwen if results["metadata"]["benchmark"]["name"] == "TinyARC"][0]["benchmark_results"]["run_output"]:
    print(entry["response"])

A
A
A
A
A
A
A
A
A
A
A
A
Okay, let's see. The question is asking which energy source is not easily replaced in nature. The options are coal, sunlight, water, and wind.First, I need to remember what each of these energy sources is. Coal is a fossil fuel, right? It's formed from the remains of plants that lived millions of years ago. Fossil fuels take a really long time to form, so they're considered non-renewable because they can't be replenished quickly. Sunlight is from the sun, which is a renewable resource because the sun is going to keep shining for a long time. Water, like in hydroelectric power, is also renewable because the water cycle keeps it moving. Wind is another renewable source since the wind is constantly generated by the sun heating the Earth's surface unevenly.So the question is about which one isn't easily replaced. Since coal takes millions of years to form, it's not something that can be replenished quickly. The others
A
A
A
A
A
A
A
A
A
A
Okay, let's see. The questio

In [273]:
os.listdir("..")

['.amlignore',
 '.amlignore.amltmp',
 '.env',
 '.flake8',
 '.git',
 '.github',
 '.gitignore',
 '.pre-commit-config.yaml',
 'azure_magic',
 'data',
 'docs',
 'emissions (1).csv',
 'emissions.csv',
 'env',
 'filter_leaderboard',
 'fuse_connection.cfg',
 'leaderboard_eurollm_large',
 'leaderboard_eurollm_large_tmp',
 'leaderboard_eurollm_small',
 'leaderboard_eurollm_small_tmp',
 'leaderboard_falcon_small',
 'leaderboard_falcon_small_tmp',
 'leaderboard_final',
 'leaderboard_gemma_large',
 'leaderboard_gemma_large_gossa',
 'leaderboard_gemma_large_tmp',
 'leaderboard_gemma_large_v2',
 'leaderboard_gemma_large_v2_tmp',
 'leaderboard_gemma_small',
 'leaderboard_gemma_small_summary',
 'leaderboard_gemma_small_summary_tmp',
 'leaderboard_gemma_small_tiny',
 'leaderboard_gemma_small_tiny_tmp',
 'leaderboard_gemma_small_tmp',
 'leaderboard_gpt4o',
 'leaderboard_gpt4o_mini',
 'leaderboard_gpt4o_mini_tmp',
 'leaderboard_gpt4o_tmp',
 'leaderboard_llama_small',
 'leaderboard_llama_small_tmp',
 'lea

# Dump results

In [51]:
%pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 13.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [36]:
os.listdir("..")
results_files = [
    'leaderboard_eurollm_large', 'leaderboard_eurollm_small',
     'leaderboard_falcon_small',
     # 'leaderboard_gemma_large',
     'leaderboard_gemma_large_gossa', 'leaderboard_gemma_small', 'leaderboard_gemma_small_summary', 'leaderboard_gemma_small_tiny',
     'leaderboard_gpt4o', 'leaderboard_gpt4o_mini',
     'leaderboard_llama_small',
     'leaderboard_mistral_tiny', 'leaderboard_mistral_small', 'leaderboard_mistral_small_summary',
     'leaderboard_olmo_large', 'leaderboard_olmo_small', 'leaderboard_olmo_large_summary',
     'leaderboard_phi_mini',
     'leaderboard_qwen_large', 'leaderboard_qwen_small', 'leaderboard_qwen_small_nothinking', 'leaderboard_qwen_large_nothinking',
     'leaderboard_tinyllama',
]
results = sum([json.loads(open(f"../{file}", "rb").read()) for file in results_files], [])

In [52]:
import pandas as pd
from collections import defaultdict

# This will collect one dataframe per benchmark
benchmark_dfs = defaultdict(dict)
benchmark_df = {}

for result in results:
    benchmark_name = result["metadata"]["benchmark"]["name"]
    model_name = result["metadata"]["llm"]["model_name"]
    
    outputs = result["benchmark_results"]["run_output"]
    for idx, sample in enumerate(outputs):
        source = sample["source"] if "source" in sample else sample["input"]
        target = sample["target"] if "target" in sample else sample["summary"]
        response = sample["response"]
        
        if source not in benchmark_dfs[benchmark_name]:
            benchmark_dfs[benchmark_name][source] = {}
            benchmark_dfs[benchmark_name][source]["source"] = source
            benchmark_dfs[benchmark_name][source]["target"] = target
        benchmark_dfs[benchmark_name][source][model_name] = response

        if source not in benchmark_df:
            benchmark_df[source] = {}
            benchmark_df[source]["target"] = target
            benchmark_df[source]["benchmark"] = benchmark_name
        benchmark_df[source][model_name] = response

# Convert to pandas DataFrames
dfs = {}
for benchmark_name, data in benchmark_dfs.items():
    df = pd.DataFrame.from_dict(data, orient='index')
    df.index.name = "source"
    dfs[benchmark_name] = df

    df.to_csv(f"{benchmark_name}.csv")

order = ["TinyMMLU", "TinyARC", "TinyTruthfulQA", "INT_Duidelijke_Taal-detailed", 
         "AmsterdamSimplification-detailed", "CNNDailyMail", "XSum"]
df = pd.DataFrame.from_dict(benchmark_df, orient='index')
df.index.name = "source"
df['benchmark'] = pd.Categorical(df['benchmark'], categories=order, ordered=True)
df = df.sort_values('benchmark')
df.to_csv(f"all_benchmarks.csv")
df.to_excel(f"all_benchmarks.xlsx")